In [12]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import os
import h5py
os.environ['PROJ_LIB'] = '/home/lty/anaconda3/envs/hloc/share/proj'

from osgeo import gdal, osr
from geopy.distance import geodesic

# from scipy.stats import pairs

from hloc import extract_features, match_features
from hloc.utils.io import get_matches, get_keypoints
from hloc.visualization import plot_images, plot_keypoints, plot_matches, read_image, add_text

from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs, extract_rotation_angle, \
    draw_fusion_keyframe_traj


In [13]:
image_dir = Path("/home/lty/datasets/RealUAV/city1/")
seu_uav_dir = image_dir / "uav"
seu_tif_dir = image_dir / "tif"
output_dir = Path("/home/lty/outputs/RealUAV/city1")
output_dir.mkdir(exist_ok=True, parents= True)
pairs_path = output_dir/"pairs.txt"
img_list_path = output_dir/"img_list.txt"
uav_list_path = output_dir/"uav_list.txt"
tif_list_path = output_dir/"tif_list.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"
loc_path = output_dir/"loc.txt"# 保存定位结果

In [14]:
image_extensions = ['.jpg', '.jpeg', '.png', '.tif', '.tiff']
# 收集 'seu_uav' 文件夹中的所有图像文件
uav_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_uav_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

# 收集 'seu_tif' 文件夹中的所有图像文件
tif_images = [
    str(p.relative_to(image_dir))  # 获取相对于 image_dir 的路径，包含文件夹前缀
    for p in seu_tif_dir.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]
uav_images = sorted(uav_images, key = sort_key)
tif_images = sorted(tif_images, key = sort_key)
# 将 UAV 和 TIF 图像文件列表合并
img_list = uav_images + tif_images
# 打印 img_list 以验证结果
# for img in img_list:
#     print(img)
#save img_list
with open(img_list_path, 'w') as f:
    for img in img_list:
        f.write(img + "\n")

with open(uav_list_path, 'w') as f:
    for img in uav_images:
        f.write(img + "\n")
with open(tif_list_path, 'w') as f:
    for img in tif_images:
        f.write(img + "\n")

In [15]:
import time
t0 = time.time()
extract_features.main(
    conf=extract_features.confs['superpoint_max'],
    image_dir=image_dir,
    image_list=img_list,
    feature_path = features_path,
)
t1 = time.time()
match_features.main(
    conf=match_features.confs["superpoint+lightglue"],
    pairs=pairs_path,
    features=features_path,
    matches=matches_path,
)
t2 = time.time()
print(f"Feature extraction time: {t1-t0:.3f}s")
print(f"Feature matching time: {t2-t1:.3f}s")

[2025/05/14 16:09:12 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2025/05/14 16:09:12 hloc INFO] Skipping the extraction.
[2025/05/14 16:09:12 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2025/05/14 16:09:12 hloc INFO] Skipping the matching.


Feature extraction time: 0.178s
Feature matching time: 0.008s


In [16]:
geotransform = []
# 打开 geotransform.txt 文件
with open("/home/lty/datasets/RealUAV/city1/geotransform.txt", "r") as f:
    # 逐行读取文件内容
    for line in f:
        # 去除行首尾的空白字符和换行符
        line = line.strip()
        if line:
            try:
                # 将字符串转换为浮点数
                value = float(line)
                # 将数值添加到列表中
                geotransform.append(value)
            except ValueError:
                print(f"无法将以下内容转换为数值：'{line}'")
                # 根据需要，可以选择跳过或停止程序
                continue
# 输出读取到的 geotransform 数据
print("Geotransform 数组：\n", geotransform)

Geotransform 数组：
 [12123218.434142068, 0.2985821417389691, 0.0, 4062398.742272254, 0.0, -0.2985821417389691]


In [25]:

pairs = read_pairs(pairs_path)
print(f"Found {len(pairs)} image pairs.")
start_x, start_y = 0, 0
x_in_map, y_in_map = 0, 0
x_origin, y_origin = 0, 0
start = time.time()
n = 0
# match_save_path = output_dir/"matches"
with open(loc_path, 'w') as loc_file:
    for img_uav, img_tif in pairs:
        print(f"UAV: {img_uav} - TIF: {img_tif}")
        kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
        matches,scores = get_matches(matches_path, img_uav, img_tif)
        print(matches.shape)
        pts1 = kp1[matches[:,0]]
        pts2 = kp2[matches[:,1]]
        
        H, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 10.0)

        print(H)
        if H is not None:
            h_uav, w_uav = 490, 490
            center_uav = np.array([[w_uav / 2, h_uav / 2]], dtype=np.float32)  
            center_uav[0][0] = center_uav[0][0]# 形状为 (1, 2)
            center_uav[0][1] = center_uav[0][1]# 形状为 (1, 2)
            # 将中心点坐标转换为齐次坐标
            center_uav_homogeneous = np.array([center_uav[0][0], center_uav[0][1], 1.0])  # 形状为 (3,)
            # 通过单应性矩阵进行变换
            center_tif_homogeneous = np.dot(H, center_uav_homogeneous)  # 形状为 (3,)
            # 归一化
            center_tif = center_tif_homogeneous[:2] / center_tif_homogeneous[2]  # 形状为 (2,)
            # center_tif = center_tif_homogeneous[:2] / 1 # 形状为 (2,)
            angle = extract_rotation_angle(H)
            print(f"旋转角度 (度): {angle}")
            # 打印结果
            print(f"无人机图像中心点在tif的位置：{center_tif}")
            pixel_x, pixel_y = center_tif
            tif_name = os.path.basename(img_tif)
            match = re.match(r"(\d+)_(\d+)_(\d+).tif", tif_name)
            if match:
                start_x = match.group(2)
                start_y = match.group(3)
                x_in_map = int(start_x) + pixel_x
                y_in_map = int(start_y) + pixel_y
                print(f"无人机图像中心点在地图上的位置：{x_in_map},{y_in_map}")
            try:
                lon, lat, x_geo, y_geo = pixel_to_geo_coordinates(x_in_map, y_in_map, geotransform)
                print(f"无人机图像中心点的经纬度：{lat}, {lon}")
                if n == 0:
                    x_origin, y_origin = x_geo, y_geo
                loc_file.write(f"{img_uav} {lon:.8f} {lat:.8f} {x_in_map:.8f} {y_in_map:.8f} {x_geo:.8f} {y_geo:.8f} {x_geo-x_origin:.10f} {y_origin-y_geo:.10f} {angle:.8f}\n")
                n+=1
            except Exception as e:
                print(f"无法计算无人机图像中心点的经纬度：{e}")
end = time.time()
print(f"Time: {end-start:.3f}s")

Found 429 image pairs.
UAV: uav/001.jpg - TIF: tif/278_1906_2356.tif
(320, 2)
[[ 1.23090714e+00 -8.69837051e-02  1.06051208e+02]
 [ 3.15878901e-03  1.18495299e+00  5.50721546e+01]
 [-1.88629940e-05 -1.31498080e-04  1.00000000e+00]]
旋转角度 (度): 2.1368743727906625
无人机图像中心点在tif的位置：[401.08791184 359.39925548]
无人机图像中心点在地图上的位置：2307.087911841699,2715.3992554817537
无人机图像中心点的经纬度：34.24381778692781, 108.9109122130888
UAV: uav/002.jpg - TIF: tif/278_1906_2356.tif
(307, 2)
[[1.52576118e+00 1.18333444e-01 2.94834950e+01]
 [1.27420190e-01 1.49921800e+00 1.45924623e+00]
 [2.38091431e-04 1.76634954e-04 1.00000000e+00]]
旋转角度 (度): 0.1721104910945579
无人机图像中心点在tif的位置：[392.41426289 363.09251119]
无人机图像中心点在地图上的位置：2298.414262888686,2719.0925111877295
无人机图像中心点的经纬度：34.24380959805862, 108.9108889485494
UAV: uav/003.jpg - TIF: tif/278_1906_2356.tif
(276, 2)
[[ 1.78111588e+00  2.84005275e-01 -3.82878805e+01]
 [ 2.06958622e-01  1.76858180e+00 -3.30939211e+01]
 [ 4.64789874e-04  5.00841347e-04  1.00000000e+00]]
旋转角度 (度

In [18]:
import cv2
from my_pkg.tools import parse_keyframe_file, draw_keyframe_trajectory, plot_traj_tif
points_traj = []
with open("/home/lty/outputs/RealUAV/city1/loc.txt", 'r') as loc_file:
    for line in loc_file:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        x_in_map = float(parts[3])   # 调整横坐标
        y_in_map = float(parts[4])  # 调整纵坐标
        print(f"{parts[0]}: {x_in_map}, {y_in_map}")
        points_traj.append((x_in_map, y_in_map))

    # 将点转换为整数坐标（像素坐标）
points_traj = [(int(x), int(y)) for x, y in points_traj]

plot_traj_tif(
    map_image_path="/home/lty/outputs/RealUAV/city1/ran_5.png",
    loc_file_path=loc_path,
    output_image_path=output_dir/"ran_10.png",
    scale_factor=1,
)

# # 绘制关键帧的单独景象匹配结果
# map = cv2.imread("/home/lty/outputs/RealUAV/city1/gt.png", cv2.IMREAD_COLOR)
# keyframe_mapping = parse_keyframe_file("/home/lty/outputs/RealUAV/city1/KeyFrameId.txt")
# map_with_traj = draw_keyframe_trajectory(map, points_traj, keyframe_mapping)
# cv2.imwrite(output_dir/"scene_match_KF.png", map_with_traj)

uav/001.jpg: 2307.08791184, 2715.39925548
uav/002.jpg: 2298.41426289, 2719.09251119
uav/003.jpg: 2284.19383777, 2720.64572492
uav/004.jpg: 2259.35298582, 2720.16174476
uav/005.jpg: 2246.85138895, 2721.51953204
uav/006.jpg: 2225.83070088, 2723.95165658
uav/007.jpg: 2211.04424991, 2725.39681392
uav/008.jpg: 2198.35831603, 2726.61145959
uav/009.jpg: 2181.11730356, 2727.65544639
uav/010.jpg: 2160.13441862, 2728.50241887
uav/011.jpg: 2149.15281578, 2734.13906348
uav/012.jpg: 2130.13076386, 2733.67694802
uav/013.jpg: 2116.45473373, 2735.39143327
uav/014.jpg: 2101.56496417, 2738.17530784
uav/015.jpg: 2080.94442284, 2734.58027595
uav/016.jpg: 2067.84125826, 2739.46773405
uav/017.jpg: 2046.91486221, 2736.67404761
uav/018.jpg: 2033.43582693, 2740.60002724
uav/019.jpg: 2017.23470874, 2741.09351594
uav/020.jpg: 2003.75644774, 2745.26978178
uav/021.jpg: 1986.9749684, 2746.36132235
uav/022.jpg: 1970.48886473, 2748.27203546
uav/023.jpg: 1956.07037474, 2749.82921675
uav/024.jpg: 1939.58252911, 2750.51

slam traj

In [20]:
#fusion traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj, draw_fusion_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city1/geoKFrame_gt.txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"scene_match_KF.png", cv2.IMREAD_COLOR)
map_with_traj = draw_fusion_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"fusion_slam_KF_gt.png", map_with_traj)

True

In [43]:
#slam traj
import cv2
from my_pkg.tools import parse_keyframe_file, draw_slam_keyframe_traj
slam_traj_path = "/home/lty/outputs/RealUAV/city1/geoKFrame(slam).txt"
def geoXY_to_pixelXY(geo_x, geo_y, geotransform):
    x = int((geo_x - geotransform[0]) / geotransform[1])
    y = int((geo_y - geotransform[3]) / geotransform[5])
    return x, y

slam_traj = []
with open(slam_traj_path, 'r') as f:
    lines = f.readlines()
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        Pixelx, Pixely = geoXY_to_pixelXY(float(parts[0]), float(parts[1]), geotransform)
        #print(f"{parts[0]}: {Pixelx}, {Pixely}")
        x_in_map = Pixelx  # 调整横坐标  seu need *0.2
        y_in_map = Pixely 
        slam_traj.append((x_in_map, y_in_map))
        
slam_traj = [(int(x), int(y)) for x, y in slam_traj]   
map = cv2.imread(output_dir/"fusion_slam_KF.png", cv2.IMREAD_COLOR)
map_with_traj = draw_slam_keyframe_traj(map, slam_traj)
cv2.imwrite(output_dir/"compare.png", map_with_traj)

True

In [ ]:
# 打开 HDF5 文件
with h5py.File(features_path, 'r') as f:
    # 获取所有数据集的名称
    print("文件中的数据集和分组结构：")
    def print_structure(name, obj):
        """递归打印 HDF5 文件的层次结构"""
        if isinstance(obj, h5py.Group):
            print(f"Group: {name}")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset: {name} - Shape: {obj.shape} - Type: {obj.dtype}")
    f.visititems(print_structure)

    # 示例：读取一个特定图像的特征
    example_image = "seu_uav/DJI_0218.JPG"  # 替换为您的图像名称
    if example_image in f:
        group = f[example_image]
        print(f"\n特定图像 '{example_image}' 的内容:")
        for key in group.keys():
            data = group[key][:]
            print(f"{key}: {data.shape} - {data.dtype}")
    else:
        print(f"图像 '{example_image}' 不存在于 HDF5 文件中。")